Agentic AI System

In [ ]:
!pip install langgraph
!pip install langchain
!pip install openai
!pip install langchain_community
!pip install huggingface_hub transformers langchain
!pip install --upgrade langchain
!pip install langchain openai
!pip install langchain openai pydantic
!pip install --upgrade langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.0/444.0 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.74
    Uninstalling langchain-core-

In [ ]:
!pip install --upgrade langgraph

In [ ]:
from typing import Dict, List, Any, Annotated
from pydantic import BaseModel
from langchain.chat_models import ChatOpenAI
from langchain.llms import HuggingFaceHub
from langgraph.graph import StateGraph
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = "API_KEY"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "API_KEY"


Defining DealState Schema with shared fields

In [ ]:
class DealState(BaseModel):
    target_financials: Annotated[Dict[str, float], "shared"]
    acquirer_financials: Annotated[Dict[str, float], "shared"]
    deal_price: Annotated[float, "shared"]
    combined_revenue: float = 0
    combined_eps: float = 0
    acquirer_eps: float = 0
    accretion_dilution_result: str = ""
    optimal_mix: str = ""
    deal_timeline_months: int = 0
    management_list: Annotated[List[str], "shared"]
    shareholder_data: Annotated[Dict[str, str], "shared"]
    employee_incentive_programs: Annotated[Dict[str, Any], "shared"]
    key_management_to_retain: List[str] = []
    incentive_risk: str = ""
    potential_conflicts: List[str] = []
    industries_involved: Annotated[List[str], "shared"]
    countries_involved: Annotated[List[str], "shared"]
    deal_value: Annotated[float, "shared"]
    antitrust_risk: str = ""
    approvals_required: List[str] = []
    party_priorities: Annotated[Dict[str, List[str]], "shared"]
    financial_flexibility: Annotated[float, "shared"]
    regulatory_constraints: Annotated[List[str], "shared"]
    negotiation_strategy: str = ""
    fallback_option: str = ""

Define Agent Functions

In [ ]:
def analyze_financial_statements(state: DealState) -> DealState:
    state.combined_revenue = state.target_financials["revenue"] + state.acquirer_financials["revenue"]
    state.combined_eps = (
        state.target_financials["net_income"] + state.acquirer_financials["net_income"]
    ) / state.combined_revenue
    state.acquirer_eps = state.acquirer_financials["eps"]
    print("Financial Model Output:", state.model_dump())
    return state

def model_accretion_dilution(state: DealState) -> DealState:
    state.accretion_dilution_result = (
        "Accretive" if state.combined_eps > state.acquirer_eps else "Dilutive"
    )
    print("Accretion/Dilution Output:", state.model_dump())
    return state

def calculate_optimal_capital_structure(state: DealState) -> DealState:
    state.optimal_mix = (
        "70% stock / 30% cash" if state.accretion_dilution_result == "Dilutive" else "50% stock / 50% cash"
    )
    state.deal_timeline_months = 6
    print("Capital Structure Output:", state.model_dump())
    return state

def analyze_stakeholders(state: DealState) -> DealState:
    conflicts = []
    if "cash" in state.shareholder_data.values() and "stock" in state.shareholder_data.values():
        conflicts.append("Mixed payout preferences")
    state.key_management_to_retain = state.management_list[:3]
    state.incentive_risk = (
        "High" if len(state.employee_incentive_programs) < 2 else "Low"
    )
    state.potential_conflicts = conflicts
    print("Stakeholders Output:", state.model_dump())
    return state

def analyze_regulatory_compliance(state: DealState) -> DealState:
    state.antitrust_risk = (
        "High" if "Technology" in state.industries_involved else "Low"
    )
    state.approvals_required = (
        ["FTC", "SEC"] if "USA" in state.countries_involved else ["EU Commission"]
    )
    print("Regulatory Output:", state.model_dump())
    return state

def generate_negotiation_strategy(state: DealState) -> DealState:
    state.negotiation_strategy = (
        "Collaborative" if "speed" in state.party_priorities.get("acquirer", []) else "Aggressive"
    )
    state.fallback_option = (
        "Use stock instead of cash"
        if state.financial_flexibility < 100_000_000 else "Offer early close premium"
    )
    print("Negotiation Output:", state.model_dump())
    return state

Loading LLMS

In [ ]:
llm_openai = ChatOpenAI(model="gpt-4o")
llm_huggingface = HuggingFaceHub(
    repo_id="google/flan-t5-large",
    model_kwargs={"temperature": 0.5, "max_new_tokens": 200},
    task="text2text-generation"
)

Building the graph

In [ ]:
graph = StateGraph(state_schema=DealState)

graph.add_node("Financial_Model_Agent", analyze_financial_statements)
graph.add_node("Accretion_Dilution_Agent", model_accretion_dilution)
graph.add_node("Capital_Structure_Agent", calculate_optimal_capital_structure)
graph.add_node("Stakeholders_Analyst_Agent", analyze_stakeholders)
graph.add_node("Regulatory_Compliance_Agent", analyze_regulatory_compliance)
graph.add_node("Negotiation_Strategy_Agent", generate_negotiation_strategy)

# NEW: Redesign edges to avoid parallel updates to the same keys
graph.add_edge("Financial_Model_Agent", "Accretion_Dilution_Agent")
graph.add_edge("Accretion_Dilution_Agent", "Capital_Structure_Agent")
graph.add_edge("Capital_Structure_Agent", "Stakeholders_Analyst_Agent")
graph.add_edge("Stakeholders_Analyst_Agent", "Regulatory_Compliance_Agent")
graph.add_edge("Regulatory_Compliance_Agent", "Negotiation_Strategy_Agent")

Main Pipeline

In [ ]:
#Adding the code to insert for people
graph.set_entry_point("Financial_Model_Agent")
final_graph = graph.compile()

target_revenue = float(input("Enter target company revenue: "))
target_net_income = float(input("Enter target company net income: "))
target_eps = float(input("Enter target company EPS: "))

acquirer_revenue = float(input("Enter acquirer company revenue: "))
acquirer_net_income = float(input("Enter acquirer company net income: "))
acquirer_eps = float(input("Enter acquirer company EPS: "))

deal_price = float(input("Enter deal price: "))


dummy_input = {
    "target_financials": {"revenue": target_revenue, "net_income": target_net_income, "eps": target_eps},
    "acquirer_financials": {"revenue": acquirer_revenue, "net_income": acquirer_net_income, "eps": acquirer_eps},
    "deal_price": deal_price,
    "management_list": ["CEO", "CFO", "CTO", "VP Sales"],
    "shareholder_data": {"retail": "cash", "institutional": "stock"},
    "employee_incentive_programs": {"RSUs": 300},
    "industries_involved": ["Technology"],
    "countries_involved": ["USA"],
    "deal_value": deal_price,
    "party_priorities": {"acquirer": ["speed", "control"]},
    "financial_flexibility": 50_000_000,
    "regulatory_constraints": ["Antitrust risk in cloud markets"]
}

deal_state = DealState(**dummy_input)
output = final_graph.invoke(deal_state)
final_output = dict(output)
print("Final Output:", final_output)
deal_state = DealState(**dummy_input)
output = final_graph.invoke(deal_state)

#Adding the graph
print("\n--- DEAL SUMMARY ---")
print(f"Accretion/Dilution Result: {output['accretion_dilution_result']}")
print(f"Optimal Capital Mix: {output['optimal_mix']}")
print(f"Deal Timeline (Months): {output['deal_timeline_months']}")
print(f"Key Management to Retain: {', '.join(output['key_management_to_retain'])}")
print(f"Incentive Risk: {output['incentive_risk']}")
print(f"Potential Stakeholder Conflicts: {', '.join(output['potential_conflicts'])}")
print(f"Antitrust Risk: {output['antitrust_risk']}")
print(f"Regulatory Approvals Required: {', '.join(output['approvals_required'])}")
print(f"Negotiation Strategy: {output['negotiation_strategy']}")
print(f"Fallback Option: {output['fallback_option']}")



Enter target company revenue: 10000000
Enter target company net income: 2500000
Enter target company EPS: 1.52
Enter acquirer company revenue: 15000000
Enter acquirer company net income: 4200000
Enter acquirer company EPS: 1.25
Enter deal price: 3000000
Financial Model Output: {'target_financials': {'revenue': 10000000.0, 'net_income': 2500000.0, 'eps': 1.52}, 'acquirer_financials': {'revenue': 15000000.0, 'net_income': 4200000.0, 'eps': 1.25}, 'deal_price': 3000000.0, 'combined_revenue': 25000000.0, 'combined_eps': 0.268, 'acquirer_eps': 1.25, 'accretion_dilution_result': '', 'optimal_mix': '', 'deal_timeline_months': 0, 'management_list': ['CEO', 'CFO', 'CTO', 'VP Sales'], 'shareholder_data': {'retail': 'cash', 'institutional': 'stock'}, 'employee_incentive_programs': {'RSUs': 300}, 'key_management_to_retain': [], 'incentive_risk': '', 'potential_conflicts': [], 'industries_involved': ['Technology'], 'countries_involved': ['USA'], 'deal_value': 3000000.0, 'antitrust_risk': '', 'appro

In this project, I built an AI system that simulates how companies evaluate and structure mergers and acquisitions (M&A) deals. M&A deal structuring means deciding whether buying another company will financially benefit the buyer, how to pay for it (using cash or stock), how to keep key employees, and how to handle legal approvals and negotiations. These decisions are important because they determine if the deal will increase profits or create risks for the buyer. My code uses a series of specialized AI agents to automatically analyze financial data, assess risks, and recommend the best deal structure. Instead of manually doing complex financial modeling and scenario planning like a human analyst would, the system takes basic input numbers about both companies and produces recommendations for financing, management retention, legal risks, and negotiation strategies. This type of analysis is normally done by teams of financial analysts, but my AI model streamlines and automates the process.